# Sales Order Detail: Bronze to Silver
Azure Synapse Analytics / Spark 3.5: append adventureworks_bronze.salesorderdetail into adventureworks_silver.salesorderdetail as Delta. The first run reads existing Bronze rows; later runs process new appends using the same persistent checkpoint.

Replace the Silver and checkpoint container placeholders in Cell 2 before running. The type mapping follows the supplied Synapse schema and retains _ingested_at and _source_file. SQL varchar maps to Spark string; SQL datetime2 maps to Spark timestamp with microsecond precision, not datetime2(7)'s 100-nanosecond precision. Null values remain allowed. Invalid casts fail under ANSI mode.

This is an alternative to the all-tables notebook for this table. It uses the same destination and per-table checkpoint path; do not run both jobs concurrently. If that stream already ran, retain its checkpoint and compatible schema. With a fresh checkpoint, use a new or empty Silver target to avoid duplicating existing history. Do not delete checkpoints between runs.

Bronze must be append-only. No updates, deletes, business-key deduplication, or automatic schema evolution are implemented. Changing casts does not recast historical Silver rows; incompatible schema changes require a planned rebuild.

In [ ]:
bronze_table = 'adventureworks_bronze.salesorderdetail'
silver_database = 'adventureworks_silver'
silver_table = f'{silver_database}.salesorderdetail'
silver_path = 'abfss://<silver-container>@datalakejdobbins.dfs.core.windows.net/azsql/saleslt/salesorderdetail'
checkpoint_path = 'abfss://<checkpoint-container>@datalakejdobbins.dfs.core.windows.net/bronze_to_silver/azsql/saleslt/salesorderdetail'

In [ ]:
storage_account = 'datalakejdobbins.dfs.core.windows.net'
linked_service = 'LS_ADLS'

spark.conf.set(
    f'spark.storage.synapse.{storage_account}.linkedServiceName',
    linked_service
)
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
hadoop_conf.set(f'fs.azure.account.auth.type.{storage_account}', 'OAuth')
hadoop_conf.set(
    f'fs.azure.account.oauth.provider.type.{storage_account}',
    'com.microsoft.azure.synapse.tokenlibrary.LinkedServiceBasedTokenProvider'
)

for container in ('bronze', 'silver'):
    print(f'Checking access to {container} using {linked_service}', flush=True)
    display(mssparkutils.fs.ls(f'abfss://{container}@{storage_account}/'))

In [ ]:
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.conf.set('spark.sql.ansi.enabled', 'true')
spark.sql(f'CREATE DATABASE IF NOT EXISTS {silver_database}')

print(f'Processing table: {bronze_table} -> {silver_table}', flush=True)

bronze = spark.readStream.format('delta').table(bronze_table)
bronze.createOrReplaceTempView('salesorderdetail_bronze_stream')

silver = spark.sql("""
    SELECT
        CAST(SalesOrderID AS INT) AS SalesOrderID,
        CAST(SalesOrderDetailID AS INT) AS SalesOrderDetailID,
        CAST(OrderQty AS INT) AS OrderQty,
        CAST(ProductID AS INT) AS ProductID,
        CAST(UnitPrice AS DECIMAL(19,4)) AS UnitPrice,
        CAST(UnitPriceDiscount AS DECIMAL(19,4)) AS UnitPriceDiscount,
        CAST(LineTotal AS DECIMAL(38,6)) AS LineTotal,
        CAST(rowguid AS STRING) AS rowguid,
        CAST(ModifiedDate AS TIMESTAMP) AS ModifiedDate,
        CAST(_ingested_at AS TIMESTAMP) AS _ingested_at,
        CAST(_source_file AS STRING) AS _source_file
    FROM salesorderdetail_bronze_stream
""")

query = (
    silver.writeStream.format('delta').outputMode('append')
    .option('path', silver_path)
    .option('checkpointLocation', checkpoint_path)
    .trigger(availableNow=True)
    .toTable(silver_table)
)
query.awaitTermination()

print(f'Completed table: {silver_table}', flush=True)